# Session 5 — Spark: clean, aggregate, join, save

Last session we **found** the problems in the messy listening log. Today we:
1. **Clean** it with Spark — the same rules as session 2, but they would work on a billion rows
2. **Aggregate**: answer questions with `groupBy` + `agg`
3. **Join**: combine plays with tracks, artists and users
4. **Save** results as CSV and Parquet, and see why data engineers prefer Parquet

## Homework review (15 min)
Open `homework/hw04_spark_basics.ipynb`.

---
## Setup

In [1]:
# Run me first: finds the course data, or downloads it when running in Google Colab.
import subprocess
from pathlib import Path

COURSE_REPO = "https://github.com/alinafazliu/data-engineering-modul-1.git"


def find_data():
    for folder in [Path.cwd(), *Path.cwd().parents, Path("/content/course")]:
        for candidate in [folder / "data", folder / "month_course" / "data"]:
            if (candidate / "reference" / "tracks.csv").exists():
                return candidate


DATA = find_data()
if DATA is None:
    subprocess.run(["git", "clone", "--depth", "1", "--quiet", COURSE_REPO, "/content/course"], check=True)
    DATA = find_data()
print("Course data:", DATA)

Course data: /content/course/data


In [2]:
# Start Spark. Colab already has PySpark; anywhere else we install it first.
import importlib.util
import os
import subprocess
import sys

if importlib.util.find_spec("pyspark") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "pyspark>=3.5,<3.6"], check=True)
os.environ["PYSPARK_PYTHON"] = sys.executable  # Spark's workers must use the same Python as the notebook

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("wavely")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.ansi.enabled", "false")  # same behaviour on Spark 3.5 and 4.x: bad casts give null
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "is ready")

Spark 4.0.5 is ready


In [4]:
log = spark.read.csv(str(DATA / "messy" / "listening_log_messy.csv"), header=True)
log.show(5, truncate=False)

+-------+-------------------+-----+--------------+-------------------+--------+-------+------+
|play_id|played_at          |user |artist        |track              |duration|country|device|
+-------+-------------------+-----+--------------+-------------------+--------+-------+------+
|1      |01/09/2026 23:56   |U0293|Calvin Harris | Little Promises   |47      |DE     |mobile|
|2      |03/09/2026 21:21   |U0025|Billie Eilish | Little Lights     |1:25    |AT     |Mobile|
|3      |2026-09-02 12:52:37|U0075|EMINEM        | Northern Thunder  |86      |SK     |mobile|
|4      |02/09/2026 18:38   |U0046|Eminem        |Broken Shadows     |0:17    |SK     |mobile|
|5      |2026-09-06 11:03:45|U0396|Kendrick Lamar|electric lights    |87      |AT     |Mobile|
+-------+-------------------+-----+--------------+-------------------+--------+-------+------+
only showing top 5 rows


---
## Part 1 — Cleaning with Spark

Spark has built-in functions for almost every cleaning task. A few you need today:

| Problem | Spark function |
|---|---|
| exact duplicate rows | `df.dropDuplicates()` (or `dropDuplicates(["col"])` for duplicates by key) |
| spaces around text | `F.trim(col)` |
| casing | `F.lower`, `F.upper`, `F.initcap` (= Python's `.title()`) |
| replace text | `F.regexp_replace(col, pattern, replacement)` |
| change type | `col.cast("int")` — gives `null` if the text is not a number |
| if / elif / else | `F.when(cond, value).when(...).otherwise(value)` |
| first non-null value | `F.coalesce(a, b, c)` |
| text → timestamp | `F.to_timestamp(col, "dd/MM/yyyy HH:mm")` — `null` if the format does not match |

Let's clean the names together:

In [5]:
names_fixed = (
    log
    .dropDuplicates()
    .withColumn("artist", F.initcap(F.trim(F.col("artist"))))
    .withColumn("track", F.initcap(F.trim(F.col("track"))))
)
names_fixed.groupBy("artist").count().orderBy("artist").show(25)

+----------------+-----+
|          artist|count|
+----------------+-----+
|           Adele|   33|
|  Arctic Monkeys|    8|
|       Bad Bunny|   35|
|   Billie Eilish|   56|
|   Calvin Harris|   15|
|        Coldplay|   19|
|       Daft Punk|   25|
|    David Guetta|   37|
|           Drake|   35|
|        Dua Lipa|   37|
|      Ed Sheeran|   20|
|          Eminem|   28|
|     Hans Zimmer|    4|
| Imagine Dragons|   29|
|  Kendrick Lamar|   36|
|Ludovico Einaudi|   26|
|           Queen|   36|
|         Rosalía|   34|
|    Taylor Swift|   44|
|      The Weeknd|   43|
+----------------+-----+



Every artist now appears once with one spelling. Compare: in session 2 this took a function and a loop.

### Exercise 1 (Core) — Country and device
Starting from `names_fixed`, build `step2` where:
- `country` is trimmed and upper-case (`" at "` → `"AT"`)
- `device` is trimmed, lower-case, and spaces are replaced by `_` (`"Smart Speaker"` → `"smart_speaker"`)

In [6]:
# YOUR CODE HERE
step2 = (
    names_fixed
    .withColumn("country", F.upper(F.trim(F.col("country"))))
    .withColumn(
        "device",
        F.regexp_replace(F.lower(F.trim(F.col("device"))), " ", "_")
    )
)

step2.groupBy("country").count().show()
step2.groupBy("device").count().show()

+-------+-----+
|country|count|
+-------+-----+
|     AT|  170|
|     CZ|   77|
|   NULL|   46|
|     HU|   55|
|     DE|   88|
|     SK|  112|
|     PL|   52|
+-------+-----+

+-------------+-----+
|       device|count|
+-------------+-----+
|smart_speaker|   43|
|          web|   53|
|       mobile|  362|
|      desktop|  142|
+-------------+-----+



### Exercise 2 (Core) — Durations and timestamps
Build `clean_log` from `step2`:

1. `seconds` (int) from `duration`:
   - contains `":"` → minutes * 60 + seconds. `F.split(col, ":")` gives an array; `.getItem(0)` is the first part.
   - ends with `"s"` → remove the `s` (`F.regexp_replace(col, "s$", "")`) and cast to int
   - otherwise → cast to int (`"unknown"` becomes `null` automatically)
2. `played_at` as a real timestamp: try the three formats with `F.coalesce`:
   `"yyyy-MM-dd HH:mm:ss"`, `"dd/MM/yyyy HH:mm"`, `"d MMM yyyy, HH:mm"`
3. `play_id` cast to int; drop the old `duration` column.

In [8]:
# YOUR CODE HERE
step2.show()
clean_log = (
    step2
    .withColumn(
        "seconds",
        F.when(
            F.col("duration").contains(":"),
            F.split("duration", ":").getItem(0).cast("int") * 60
            + F.split("duration", ":").getItem(1).cast("int")
        )
        .when(
            F.col("duration").endswith("s"),
            F.regexp_replace("duration", "s$", "").cast("int")
        )
        .otherwise(F.col("duration").cast("int"))
    )
    .withColumn(
        "played_at",
        F.coalesce(
            F.to_timestamp("played_at", "yyyy-MM-dd HH:mm:ss"),
            F.to_timestamp("played_at", "dd/MM/yyyy HH:mm"),
            F.to_timestamp("played_at", "d MMM yyyy, HH:mm")
        )
    )
    .withColumn("play_id", F.col("play_id").cast("int"))
    .drop("duration")
)

clean_log.printSchema()
clean_log.show(5)

print(clean_log.count(), "rows")
print(clean_log.filter(F.col("played_at").isNull()).count(), "unparsed timestamps")
clean_log.printSchema()
clean_log.show(5)
print(clean_log.count(), "rows")  # expected: 600
print(clean_log.filter(F.col("played_at").isNull()).count(), "unparsed timestamps")  # expected: 0

+-------+-------------------+-----+----------------+----------------+--------+-------+-------------+
|play_id|          played_at| user|          artist|           track|duration|country|       device|
+-------+-------------------+-----+----------------+----------------+--------+-------+-------------+
|      9|   03/09/2026 23:30|U0404|         Rosalía|   Silver Hearts|    147s|     AT|       mobile|
|     10|  4 Sep 2026, 12:00|U0397|  Kendrick Lamar|  Little Skyline|      57|     PL|       mobile|
|     14|2026-09-05 12:53:53|U0584|      The Weeknd|    Neon Highway|    2:03|     AT|       mobile|
|     62|2026-09-01 22:05:34|U0178|       Bad Bunny| Endless Thunder|     153|     SK|          web|
|     66|   07/09/2026 09:12|U0288|Ludovico Einaudi|     Velvet Fire|      24|     HU|       mobile|
|     75|   04/09/2026 09:07|U0223|    David Guetta|   Silver Echoes|     94s|     SK|      desktop|
|     76|2026-09-03 06:42:42|U0730|    Taylor Swift|Electric Skyline|    0:33|     HU|     

> **Under the hood — why not reuse our Python function?**
> Spark *can* run normal Python functions on every row (a **UDF**, user-defined function):
> ```python
> parse_duration_udf = F.udf(parse_duration, "int")
> log.withColumn("seconds", parse_duration_udf("duration"))
> ```
> It works, but every value must be shipped from Spark's engine (running on the Java Virtual Machine) to a Python process and back.
> Built-in functions run inside the engine and are usually **much** faster. Rule of thumb: built-ins first, UDFs only when there is no other way.

---
## Part 2 — Aggregations

`groupBy(...)` + `agg(...)` = "for each group, calculate ...". Many calculations at once, each with a name (`alias`):

In [9]:
(
    clean_log
    .groupBy("device")
    .agg(
        F.count("*").alias("plays"),
        F.countDistinct("user").alias("listeners"),
        F.round(F.avg("seconds"), 1).alias("avg_seconds"),
    )
    .orderBy(F.desc("plays"))
    .show()
)

+-------------+-----+---------+-----------+
|       device|plays|listeners|avg_seconds|
+-------------+-----+---------+-----------+
|       mobile|  362|      260|      116.4|
|      desktop|  142|      101|      116.2|
|          web|   53|       43|      118.8|
|smart_speaker|   43|       33|      111.7|
+-------------+-----+---------+-----------+



Useful aggregate functions: `F.count`, `F.countDistinct`, `F.sum`, `F.avg`, `F.min`, `F.max`.
They all **ignore nulls** (except `F.count("*")`, which counts rows).

### Exercise 3 (Core) — Top artists again
Compute the **top 5 artists by listening minutes** (`sum(seconds) / 60`, rounded to 1 decimal) and their number of plays.
Do you get the same answer as in session 2, Exercise 7?

In [54]:
# YOUR CODE HERE
clean_log.show()
top_artists = (
    clean_log
    .groupBy('artist')
    .agg(
        F.round(F.sum('seconds')/60, 1).alias('listening_minutes'),
        F.count('play_id').alias('number_of_plays')
    )
    .orderBy(F.desc('listening_minutes'))
    .limit(5)
)

top_artists.show()

+-------+-------------------+-----+----------------+----------------+-------+-------------+-------+
|play_id|          played_at| user|          artist|           track|country|       device|seconds|
+-------+-------------------+-----+----------------+----------------+-------+-------------+-------+
|      9|2026-09-03 23:30:00|U0404|         Rosalía|   Silver Hearts|     AT|       mobile|    147|
|     10|2026-09-04 12:00:00|U0397|  Kendrick Lamar|  Little Skyline|     PL|       mobile|     57|
|     14|2026-09-05 12:53:53|U0584|      The Weeknd|    Neon Highway|     AT|       mobile|    123|
|     62|2026-09-01 22:05:34|U0178|       Bad Bunny| Endless Thunder|     SK|          web|    153|
|     66|2026-09-07 09:12:00|U0288|Ludovico Einaudi|     Velvet Fire|     HU|       mobile|     24|
|     75|2026-09-04 09:07:00|U0223|    David Guetta|   Silver Echoes|     SK|      desktop|     94|
|     76|2026-09-03 06:42:42|U0730|    Taylor Swift|Electric Skyline|     HU|       mobile|     33|


**Stretch:** listening minutes per **day** (`F.to_date("played_at")`) and country. Which country listened most on 5 September?

In [63]:
# YOUR CODE HERE

daily_country = (
    clean_log
    .withColumn("play_date", F.to_date("played_at"))
    .groupBy("play_date", "country")
    .agg(
        F.round(F.sum("seconds") / 60, 1).alias("minutes")
    )
    .orderBy("play_date", F.desc("minutes"))
)

daily_country.show()
daily_country.filter(
    F.col("play_date") == "2026-09-05"
).orderBy(F.desc("minutes")).limit(1).show()

+----------+-------+-------+
| play_date|country|minutes|
+----------+-------+-------+
|2026-09-01|     AT|   39.0|
|2026-09-01|     SK|   28.7|
|2026-09-01|   NULL|   27.9|
|2026-09-01|     CZ|   25.5|
|2026-09-01|     DE|   25.2|
|2026-09-01|     HU|   17.5|
|2026-09-01|     PL|   13.7|
|2026-09-02|     AT|   42.8|
|2026-09-02|     SK|   28.9|
|2026-09-02|     DE|   22.7|
|2026-09-02|   NULL|   20.1|
|2026-09-02|     CZ|   13.3|
|2026-09-02|     HU|   13.2|
|2026-09-02|     PL|    7.8|
|2026-09-03|     AT|   50.4|
|2026-09-03|     SK|   42.5|
|2026-09-03|     DE|   21.0|
|2026-09-03|     PL|   16.2|
|2026-09-03|   NULL|    8.6|
|2026-09-03|     CZ|    7.3|
+----------+-------+-------+
only showing top 20 rows
+----------+-------+-------+
| play_date|country|minutes|
+----------+-------+-------+
|2026-09-05|     AT|   50.8|
+----------+-------+-------+



---
## Part 3 — Joins

The real Wavely event stream only contains **ids** (`user_id`, `track_id`). Names, genres and plans live in separate
**reference** tables. A **join** combines them by matching keys.

```
plays                         tracks                               plays ⨝ tracks
event_id  track_id            track_id  title       genre          event_id  track_id  title       genre
e1        T0028       ⨝       T0028     Neon Rain   pop      =     e1        T0028     Neon Rain   pop
e2        T0128               T0128     Wild Fire   rock           e2        T0128     Wild Fire   rock
```

Let's load the first week of real play events (one file per day — Spark reads them all with a `*` pattern) and the reference tables.

In [64]:
PLAYS_SCHEMA = (
    "event_id STRING, event_ts TIMESTAMP, user_id STRING, track_id STRING, ms_played INT, "
    "device STRING, country STRING, skipped BOOLEAN, shuffle BOOLEAN"
)
plays = (
    spark.read.csv(str(DATA / "raw" / "plays_2026-09-0[1-7].csv"), header=True, schema=PLAYS_SCHEMA)
    .dropDuplicates(["event_id"])  # the app sometimes sends an event twice
    .filter(F.col("ms_played") >= 0)  # remove broken rows (more on this in session 6)
)

tracks = spark.read.csv(str(DATA / "reference" / "tracks.csv"), header=True, inferSchema=True)
artists = spark.read.csv(str(DATA / "reference" / "artists.csv"), header=True, inferSchema=True)
users = spark.read.csv(str(DATA / "reference" / "users.csv"), header=True, inferSchema=True)

print(plays.count(), "plays")
plays.show(3)

30395 plays
+------------+-------------------+-------+--------+---------+-------+-------+-------+-------+
|    event_id|           event_ts|user_id|track_id|ms_played| device|country|skipped|shuffle|
+------------+-------------------+-------+--------+---------+-------+-------+-------+-------+
|0000cf6f6b8e|2026-09-05 17:26:03|  U0348|   T0007|    13634|desktop|     CZ|   true|   true|
|00056eafa31d|2026-09-04 17:09:53|  U0782|   T0101|     4452| mobile|     SK|   true|  false|
|0007b5bc3a22|2026-09-01 17:40:56|  U0276|   T0129|    16929| mobile|     CZ|   true|   true|
+------------+-------------------+-------+--------+---------+-------+-------+-------+-------+
only showing top 3 rows


In [65]:
plays_with_tracks = plays.join(tracks, on="track_id", how="inner")
plays_with_tracks.select("event_id", "track_id", "title", "genre", "ms_played").show(5)

+------------+--------+--------------+----------+---------+
|    event_id|track_id|         title|     genre|ms_played|
+------------+--------+--------------+----------+---------+
|0000cf6f6b8e|   T0007|   Summer City|       pop|    13634|
|00056eafa31d|   T0101|   Neon Dreams|   hip-hop|     4452|
|0007b5bc3a22|   T0129|  Lonely River|electronic|    16929|
|000adec8e137|   T0031|Electric Stars|       pop|    33746|
|00106b011fea|   T0013|  Little Stars|       pop|   201000|
+------------+--------+--------------+----------+---------+
only showing top 5 rows


Join types you need to know:

| `how=` | keeps |
|---|---|
| `"inner"` | only rows with a match on **both** sides |
| `"left"` | **all** rows of the left table; `null`s where the right side has no match |
| `"left_anti"` | left rows **without** a match — perfect for finding orphans ("plays of tracks we don't know") |

Careful with columns that exist on both sides: `tracks` and `artists` both have a `genre` column, and
`users` and `plays` both have `country`. Select or rename before joining to avoid confusion.

In [66]:
orphans = plays.join(tracks, on="track_id", how="left_anti")
print(orphans.count(), "plays with an unknown track")

0 plays with an unknown track


### Exercise 4 (Core) — Skip rate per genre
Which genre gets skipped most? Join `plays` with `tracks` and compute per genre:
`plays` (count) and `skip_rate` = average of `skipped` cast to int, in percent, rounded to 1 decimal. Sort by skip rate.

In [67]:
# YOUR CODE HERE
skip_by_genre = (
    plays.join(tracks, on="track_id", how="inner")
    .groupBy("genre")
    .agg(
        F.count("*").alias("plays"),
        F.round(
            F.avg(F.col("skipped").cast("int")) * 100, 1
        ).alias("skip_rate")
    )
    .orderBy(F.desc("skip_rate"))
)

skip_by_genre.show()

+----------+-----+---------+
|     genre|plays|skip_rate|
+----------+-----+---------+
|   hip-hop| 5054|     35.9|
|     latin| 3352|     32.2|
|       pop|11346|     30.7|
|electronic| 3779|     26.8|
|      rock| 5076|     24.3|
| classical| 1788|     16.8|
+----------+-----+---------+



### Exercise 5 (Core) — Free vs premium
Do premium users listen more? Join `plays` with `users` (only `user_id` and `plan` — use `users.select(...)`) and compute per plan:
- `listeners`: distinct users
- `hours`: total `ms_played` in hours (rounded to 1 decimal)
- `hours_per_listener`: hours / listeners (rounded to 2 decimals)

In [68]:
# YOUR CODE HERE
by_plan = (
    plays
    .join(
        users.select("user_id", "plan"),
        on="user_id",
        how="inner"
    )
    .groupBy("plan")
    .agg(
        F.countDistinct("user_id").alias("listeners"),
        F.round(
            F.sum("ms_played") / 3600000, 1
        ).alias("hours")
    )
    .withColumn(
        "hours_per_listener",
        F.round(F.col("hours") / F.col("listeners"), 2)
    )
)

by_plan.show()

+-------+---------+-----+------------------+
|   plan|listeners|hours|hours_per_listener|
+-------+---------+-----+------------------+
|   free|      455|515.0|              1.13|
|premium|      344|724.1|               2.1|
+-------+---------+-----+------------------+



### Exercise 6 (Stretch) — Top artist per country
Join plays → tracks → artists. For each listener `country`, find the artist with the most plays.
Hint: first count plays per (country, artist_name). Then either use a **window function** (look up `Window.partitionBy(...).orderBy(...)` and `F.row_number()`)
or think of another way.

In [ ]:
# YOUR CODE HERE

---
## Part 4 — Saving results: CSV vs Parquet

Spark writes a **folder** (not one file): one part-file per partition, so every executor can write in parallel.
`mode("overwrite")` replaces old results — so running the cell twice gives the same result.

In [69]:
OUTPUT = DATA.parent / "output"

clean_log.write.mode("overwrite").option("header", True).csv(str(OUTPUT / "clean_log_csv"))
clean_log.write.mode("overwrite").parquet(str(OUTPUT / "clean_log_parquet"))

for folder in ["clean_log_csv", "clean_log_parquet"]:
    files = [p for p in (OUTPUT / folder).iterdir() if p.name.startswith("part-")]
    size_kb = sum(p.stat().st_size for p in files) / 1024
    print(f"{folder:<20} {len(files)} part file(s), {size_kb:.1f} KB")

clean_log_csv        1 part file(s), 43.2 KB
clean_log_parquet    1 part file(s), 16.6 KB


Now read both back and compare the schemas:

In [70]:
spark.read.csv(str(OUTPUT / "clean_log_csv"), header=True).printSchema()
spark.read.parquet(str(OUTPUT / "clean_log_parquet")).printSchema()

root
 |-- play_id: string (nullable = true)
 |-- played_at: string (nullable = true)
 |-- user: string (nullable = true)
 |-- artist: string (nullable = true)
 |-- track: string (nullable = true)
 |-- country: string (nullable = true)
 |-- device: string (nullable = true)
 |-- seconds: string (nullable = true)

root
 |-- play_id: integer (nullable = true)
 |-- played_at: timestamp (nullable = true)
 |-- user: string (nullable = true)
 |-- artist: string (nullable = true)
 |-- track: string (nullable = true)
 |-- country: string (nullable = true)
 |-- device: string (nullable = true)
 |-- seconds: integer (nullable = true)



| | CSV | Parquet |
|---|---|---|
| readable by humans | yes | no (binary) |
| keeps column types | no — everything is text again | **yes** |
| size | large | small (compressed) |
| read only some columns | must read everything | reads only the columns you need (**columnar**) |
| used for | exchanging data with people / old systems | **data lakes, pipelines, Spark** |

> **Under the hood:** `df.write.partitionBy("country").parquet(path)` creates one sub-folder per country (`country=AT/`, `country=SK/`, ...).
> A query that filters on `country = 'AT'` then only reads that folder. You will use this in session 9.

## Wrap-up

| Concept | Example |
|---|---|
| clean | `dropDuplicates`, `trim`, `initcap`, `regexp_replace`, `cast`, `F.when`, `F.coalesce`, `to_timestamp` |
| aggregate | `groupBy("x").agg(F.sum("y").alias("total"))` |
| join | `a.join(b, on="key", how="left")`, `left_anti` for orphans |
| save | `df.write.mode("overwrite").parquet(path)` |

**Homework:** `homework/hw05_spark_analysis.ipynb`